# Day 043 — Exercise 5: ask_db

**What you'll build:** `ask_db(conn, question, model='llama3.2') -> str` — the full natural-language-to-SQL pipeline: schema → prompt → LLM → extract SQL → validate → execute → return string result.

**Why it matters:** This is the Day 43 deliverable — a function that lets anyone query a database in plain English. It composes everything from today: schema description, prompt construction, SQL extraction, safety checking, and query execution. Day 43 is to SQL what Day 41's `ask_df` is to pandas.

## Provided: All Helper Functions

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sqlite3
import re
import ollama


import sqlite3

def setup_db(conn):
    cur = conn.cursor()
    cur.execute('''
        CREATE TABLE IF NOT EXISTS orders (
            order_id  INTEGER PRIMARY KEY,
            product   TEXT,
            category  TEXT,
            region    TEXT,
            price     REAL,
            quantity  INTEGER,
            revenue   REAL
        )''')
    cur.execute('''
        CREATE TABLE IF NOT EXISTS products (
            product    TEXT PRIMARY KEY,
            category   TEXT,
            unit_price REAL
        )''')
    rows = [
        (1,'Widget','Electronics','North',25.0,10,250.0),
        (2,'Gadget','Electronics','South',150.0,3,450.0),
        (3,'Widget','Electronics','South',25.0,5,125.0),
        (4,'Doohickey','Accessories','East',8.0,50,400.0),
        (5,'Gadget','Electronics','East',150.0,7,1050.0),
        (6,'Widget','Electronics','East',25.0,4,100.0),
        (7,'Doohickey','Accessories','North',8.0,20,160.0),
        (8,'Gadget','Electronics','North',150.0,2,300.0),
        (9,'Widget','Electronics','West',25.0,6,150.0),
        (10,'Doohickey','Accessories','South',8.0,15,120.0),
        (11,'Thingamajig','Accessories','North',200.0,1,200.0),
        (12,'Thingamajig','Accessories','East',200.0,4,800.0),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO orders VALUES (?,?,?,?,?,?,?)', rows
    )
    products = [
        ('Widget','Electronics',25.0),
        ('Gadget','Electronics',150.0),
        ('Doohickey','Accessories',8.0),
        ('Thingamajig','Accessories',200.0),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO products VALUES (?,?,?)', products
    )
    conn.commit()


def run_query(conn, sql, params=()):
    cur = conn.cursor()
    cur.execute(sql, params)
    cols = [col[0] for col in cur.description]
    return [dict(zip(cols, row)) for row in cur.fetchall()]


def get_db_schema(conn) -> str:
    cur = conn.cursor()
    cur.execute(
        "SELECT name, sql FROM sqlite_master WHERE type='table' ORDER BY name"
    )
    rows = cur.fetchall()
    if not rows:
        return 'No tables found.'
    parts = []
    for name, ddl in rows:
        parts.append(f'Table: {name}')
        parts.append(ddl)
        parts.append('')
    return '\n'.join(parts).strip()


def build_sql_prompt(question: str, schema_str: str) -> str:
    return (
        'You are a SQL expert. Write a SQLite SELECT query to answer the question.\n\n'
        'Requirements:\n'
        '- Use only SELECT statements.\n'
        '- The database schema is provided below.\n'
        '- Respond with ONLY a fenced SQL code block, no explanation.\n\n'
        f'Schema:\n{schema_str}\n\n'
        f'Question: {question}'
    )


import re

def extract_sql(response: str) -> str:
    fence = '`' * 3
    match = re.search(fence + r'sql\s*(.*?)' + fence, response, re.DOTALL)
    if match:
        return match.group(1).strip()
    match = re.search(fence + r'\s*(.*?)' + fence, response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return response.strip()


import re

def is_safe_sql(sql: str) -> bool:
    normalized = re.sub(r'--[^\n]*', '', sql)
    normalized = re.sub(r'/\*.*?\*/', '', normalized, flags=re.DOTALL)
    normalized = normalized.strip().lower()
    if not normalized.startswith('select'):
        return False
    if ';' in normalized:
        return False
    return True

In [ ]:
conn = sqlite3.connect(':memory:')
setup_db(conn)

## Your Implementation

In [ ]:
def ask_db(conn, question: str, model: str = 'llama3.2') -> str:
    """
    Answer a natural-language question by generating and running SQL.

    Pipeline:
    1. get_db_schema(conn)                → schema string
    2. build_sql_prompt(question, schema)  → prompt string
    3. ollama.chat(model, messages=[...])  → LLM response
    4. extract_sql(response content)       → SQL string
    5. is_safe_sql(sql) — if False return rejection message
    6. run_query(conn, sql)               → list[dict]
    7. return str(rows) or 'No results found.'
       wrap run_query in try/except and return 'Query error: {e}' on failure
    """
    import ollama
    # TODO: schema = get_db_schema(conn)
    # TODO: prompt = build_sql_prompt(question, schema)
    # TODO: resp = ollama.chat(model=model,
    # TODO:                    messages=[{'role': 'user', 'content': prompt}])
    # TODO: sql = extract_sql(resp['message']['content'])
    # TODO: if not is_safe_sql(sql): return f'Unsafe SQL rejected: {sql[:120]}'
    # TODO: try: rows = run_query(conn, sql)
    # TODO: except Exception as e: return f'Query error: {e}'
    # TODO: if not rows: return 'No results found.'
    # TODO: return str(rows)
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'ask_db' in globals()
        passed += 1; print('\u2705 Check 1: ask_db is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: returns a string
    try:
        result = ask_db(conn, 'How many orders are there?')
        assert isinstance(result, str), \
            f'expected str, got {type(result).__name__}'
        assert len(result) > 0, 'result is empty string'
        passed += 1; print(f'\u2705 Check 2: returns a string ({result[:60]}...)')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: unsafe SQL is blocked (is_safe_sql used)
    try:
        # Test that the guardrail works by patching extract_sql
        _orig = extract_sql
        def _always_drop(resp): return 'DROP TABLE orders'
        import builtins; _g = globals()
        _g['extract_sql'] = _always_drop
        try:
            guarded = ask_db(conn, 'drop everything')
        finally:
            _g['extract_sql'] = _orig
        assert 'reject' in guarded.lower() or 'unsafe' in guarded.lower(), \
            f'unsafe SQL should be rejected, got: {guarded}'
        passed += 1; print('\u2705 Check 3: unsafe SQL is rejected')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: handles bad SQL gracefully (returns error string not exception)
    try:
        _orig = extract_sql
        def _bad_select(resp): return 'SELECT nonexistent_col FROM orders'
        _g = globals(); _g['extract_sql'] = _bad_select
        try:
            err = ask_db(conn, 'any question')
        finally:
            _g['extract_sql'] = _orig
        assert isinstance(err, str), 'error path should return string'
        passed += 1; print(f'\u2705 Check 4: bad SQL returns error string ({err[:50]})')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: pipeline produces meaningful result for revenue question
    try:
        r = ask_db(conn, 'What is the total revenue from all orders?')
        assert isinstance(r, str) and len(r) > 0
        passed += 1; print(f'\u2705 Check 5: revenue question answered ({r[:80]})')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import ollama

def ask_db(conn, question: str, model: str = 'llama3.2') -> str:
    schema = get_db_schema(conn)
    prompt = build_sql_prompt(question, schema)
    resp   = ollama.chat(model=model,
                         messages=[{'role': 'user', 'content': prompt}])
    sql    = extract_sql(resp['message']['content'])
    if not is_safe_sql(sql):
        return f'Unsafe SQL rejected: {sql[:120]}'
    try:
        rows = run_query(conn, sql)
    except Exception as e:
        return f'Query error: {e}'
    if not rows:
        return 'No results found.'
    return str(rows)
```

</details>